# CSE Dataset — Data Exploration

Exploratory analysis of the Colombo Stock Exchange unified dataset.

**Covers:**
- Dataset shape, date range, symbols
- Sector breakdown
- Trading activity over time
- Return distributions
- Correlation with global indices and macro

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
df = pd.read_parquet('../data/published/cse_unified.parquet')
df['date'] = pd.to_datetime(df['date'])
meta = pd.read_csv('../data/processed/company_metadata.csv')

print(f'Shape:        {df.shape}')
print(f'Symbols:      {df["symbol"].nunique()}')
print(f'Date range:   {df["date"].min().date()}  to  {df["date"].max().date()}')
print(f'Columns:      {df.columns.tolist()}')

## 1. Sector Breakdown

In [ ]:
if 'sector' in meta.columns:
    sector_counts = meta['sector'].value_counts()
    fig, ax = plt.subplots(figsize=(10, 5))
    sector_counts.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_xlabel('Number of companies')
    ax.set_title('CSE Companies by Sector')
    plt.tight_layout()
    plt.show()
    print(sector_counts.to_string())

## 2. Market Trading Activity Over Time

In [ ]:
trading = df[df['is_trading_day']].copy()
daily_turnover = trading.groupby('date')['turnover'].sum() / 1e9  # billions LKR
daily_active   = trading.groupby('date')['symbol'].nunique()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1.plot(daily_turnover.index, daily_turnover.values, lw=0.8, color='steelblue')
ax1.set_ylabel('Total turnover (LKR bn)')
ax1.set_title('CSE Daily Market Turnover')

ax2.plot(daily_active.index, daily_active.values, lw=0.8, color='darkorange')
ax2.set_ylabel('Active symbols')
ax2.set_title('Number of Symbols Trading Per Day')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()

## 3. Return Distribution

In [ ]:
returns = df['return_1d'].dropna()
returns_clipped = returns.clip(-0.15, 0.15)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(returns_clipped, bins=200, color='steelblue', edgecolor='none')
axes[0].set_xlabel('Daily return')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Daily Returns (all stocks)')

axes[1].hist(returns_clipped, bins=200, color='steelblue', edgecolor='none', cumulative=True, density=True)
axes[1].set_xlabel('Daily return')
axes[1].set_ylabel('Cumulative probability')
axes[1].set_title('CDF of Daily Returns')

plt.tight_layout()
plt.show()

print('Return statistics:')
print(returns.describe().round(4))
print(f'Skewness: {returns.skew():.3f}')
print(f'Kurtosis: {returns.kurtosis():.3f}')

## 4. Top 20 Stocks by Average Daily Turnover

In [ ]:
avg_turnover = (
    df[df['is_trading_day']]
    .groupby('symbol')['turnover']
    .mean()
    .sort_values(ascending=False)
    .head(20)
    / 1e6
)

fig, ax = plt.subplots(figsize=(10, 5))
avg_turnover.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Average daily turnover (LKR mn)')
ax.set_title('Top 20 Stocks by Average Daily Turnover')
plt.tight_layout()
plt.show()

## 5. Macro Correlation — Market Returns vs S&P 500 / USD/LKR

In [ ]:
if 'sp500' in df.columns:
    market_ret = df.groupby('date')['return_1d'].mean().rename('cse_avg_return')
    sp500_ret  = df[['date','sp500']].drop_duplicates().set_index('date')['sp500'].pct_change().rename('sp500_return')
    combined = pd.concat([market_ret, sp500_ret], axis=1).dropna()

    corr = combined.corr().iloc[0, 1]
    print(f'Correlation between CSE avg daily return and S&P 500 daily return: {corr:.3f}')

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(combined['sp500_return'], combined['cse_avg_return'],
               alpha=0.2, s=8, color='steelblue')
    ax.set_xlabel('S&P 500 daily return')
    ax.set_ylabel('CSE average daily return')
    ax.set_title(f'CSE vs S&P 500 Daily Returns  (corr={corr:.3f})')
    plt.tight_layout()
    plt.show()

## 6. Volatility Regime — Rolling 20-day Volatility (Market Average)

In [ ]:
avg_vol = df.groupby('date')['volatility_20d'].mean()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(avg_vol.index, avg_vol.values, lw=0.9, color='crimson')
ax.set_ylabel('Avg volatility_20d')
ax.set_title('CSE Market — Average 20-day Rolling Volatility')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()